# Task 2 — Credit Card Fraud Detection
### SCT Machine Learning Internship

**Objective:** Classify credit-card transactions as fraudulent or legitimate using machine learning.

**Workflow:** Load data → Explore class imbalance → Prepare features → Train Logistic Regression, Decision Tree, and Random Forest → Evaluate fraud-focused metrics → Save results.

**Dataset location:** `C:\Users\vaibh\Downloads\CODESOFT_02\data\fraudTrain`

This notebook supports either a CSV file at that location (for example `fraudTrain.csv`) or a folder containing the CSV file. It is designed for the common transaction dataset with a target such as `is_fraud` or `Class`.


In [1]:
# 1. Imports and configuration
from pathlib import Path
import json
import re
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, average_precision_score, classification_report,
    ConfusionMatrixDisplay, confusion_matrix, f1_score,
    precision_score, recall_score, roc_auc_score,
    PrecisionRecallDisplay, RocCurveDisplay
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

warnings.filterwarnings("ignore", category=FutureWarning)

RANDOM_STATE = 42
TEST_SIZE = 0.20

# Use the exact folder path you provided.
DATA_PATH = Path(r"C:\Users\vaibh\Downloads\CODESOFT_02\data\fraudTrain")

# The common fraudTrain dataset can be very large. Set to None to use all rows.
# For a quicker first run, 200_000 is a practical starting sample size.
MAX_ROWS = 200_000

OUTPUT_DIR = Path("results")
MODEL_DIR = Path("artifacts")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print("Data path:", DATA_PATH)
print("Results will be saved to:", OUTPUT_DIR.resolve())


Data path: C:\Users\vaibh\Downloads\CODESOFT_02\data\fraudTrain
Results will be saved to: C:\Users\vaibh\Downloads\CODESOFT_02\notebooks\results


## 2. Load the dataset

The path can point directly to a `.csv` file or to a folder. If it is a folder, the notebook searches for `fraudTrain.csv` first and then for other CSV files. If multiple unrelated CSV files are present, update `DATA_PATH` to point directly to the intended file.


In [2]:
# 2. Load data from a CSV file or a folder
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset path does not exist: {DATA_PATH}\n"
        "Check the folder and filename. If your file is fraudTrain.csv, "
        "you can set DATA_PATH to that complete CSV path."
    )

if DATA_PATH.is_file():
    csv_path = DATA_PATH
else:
    preferred = list(DATA_PATH.glob("fraudTrain.csv")) + list(DATA_PATH.glob("FraudTrain.csv"))
    csv_files = preferred or sorted(DATA_PATH.glob("*.csv"))
    if not csv_files:
        raise FileNotFoundError(f"No CSV files found inside: {DATA_PATH}")
    csv_path = csv_files[0]
    if len(csv_files) > 1:
        print("CSV files found:", [p.name for p in csv_files])
        print("Using:", csv_path.name, "| Change DATA_PATH to a specific file if needed.")

print("Loading:", csv_path)
df = pd.read_csv(csv_path, nrows=MAX_ROWS)
print("Loaded shape:", df.shape)
display(df.head())


FileNotFoundError: Dataset path does not exist: C:\Users\vaibh\Downloads\CODESOFT_02\data\fraudTrain
Check the folder and filename. If your file is fraudTrain.csv, you can set DATA_PATH to that complete CSV path.

## 3. Exploratory data analysis

Fraud datasets are usually highly imbalanced: legitimate transactions greatly outnumber fraudulent ones. Accuracy alone can therefore be misleading. First identify the target column and inspect the label distribution.


In [ ]:
# 3. Dataset overview and fraud distribution
df.columns = [str(c).strip() for c in df.columns]
print("Rows and columns:", df.shape)
print("\nColumns:")
print(df.columns.tolist())
print("\nMissing values (top 20):")
display(df.isna().sum().sort_values(ascending=False).head(20).to_frame("missing_count"))

target_candidates = ["is_fraud", "Class", "class", "fraud", "isFraud", "is_fraudulent"]
target_col = next((c for c in target_candidates if c in df.columns), None)
if target_col is None:
    raise ValueError(
        "Could not identify the target column. Expected one of: "
        + ", ".join(target_candidates)
        + ". Rename your target column or update target_candidates."
    )

print("Target column:", target_col)
print("\nTarget counts:")
display(df[target_col].value_counts(dropna=False).rename("count").to_frame())
print("\nTarget proportions:")
display((df[target_col].value_counts(normalize=True, dropna=False) * 100)
        .rename("percentage").round(4).to_frame())

ax = df[target_col].value_counts().sort_index().plot(
    kind="bar", figsize=(7, 4), title="Transaction Class Distribution"
)
ax.set_xlabel("Target label (0 = legitimate, 1 = fraud)")
ax.set_ylabel("Number of transactions")
plt.tight_layout()
plt.show()


## 4. Prepare features and target

The notebook removes direct identifiers such as transaction numbers and names. When present, transaction timestamps are converted into hour, day, month, and weekday features; date of birth is converted to an approximate age. Numeric missing values are imputed, and categorical variables are imputed and one-hot encoded inside the model pipeline.

For very large files, `MAX_ROWS` controls the number of rows loaded. Sampling the first rows is convenient for a first run but is not a substitute for a representative production evaluation.


In [ ]:
# 4. Feature engineering and leakage-safe preparation
data = df.copy()
y_raw = data[target_col]

# Convert common target formats to binary 0/1.
if y_raw.dtype == "object":
    normalized = y_raw.astype(str).str.strip().str.lower()
    mapping = {
        "0": 0, "1": 1, "legitimate": 0, "genuine": 0, "normal": 0,
        "fraud": 1, "fraudulent": 1, "true": 1, "false": 0
    }
    mapped = normalized.map(mapping)
    if mapped.isna().any():
        raise ValueError("Target labels could not be mapped to 0/1. Inspect df[target_col].value_counts().")
    y = mapped.astype(int)
else:
    y = pd.to_numeric(y_raw, errors="raise").astype(int)

if set(y.unique()) != {0, 1}:
    raise ValueError(f"Expected binary target labels 0 and 1; found {sorted(y.unique())}")

X = data.drop(columns=[target_col])

# Parse transaction time and derive useful calendar features.
datetime_candidates = [c for c in X.columns if c.lower() in
                       {"trans_date_trans_time", "transaction_time", "transaction_datetime", "timestamp"}]
for col in datetime_candidates:
    dt = pd.to_datetime(X[col], errors="coerce")
    X[f"{col}_hour"] = dt.dt.hour
    X[f"{col}_day"] = dt.dt.day
    X[f"{col}_month"] = dt.dt.month
    X[f"{col}_dayofweek"] = dt.dt.dayofweek
    X = X.drop(columns=[col])

# Derive age from date of birth when both DOB and a transaction date are available.
dob_cols = [c for c in X.columns if c.lower() in {"dob", "date_of_birth", "birth_date"}]
if dob_cols:
    dob = pd.to_datetime(X[dob_cols[0]], errors="coerce")
    trans_date_col = next((c for c in data.columns if c.lower() == "trans_date_trans_time"), None)
    if trans_date_col:
        trans_dt = pd.to_datetime(data[trans_date_col], errors="coerce")
        X["customer_age"] = ((trans_dt - dob).dt.days / 365.25).clip(lower=0, upper=120)
    X = X.drop(columns=dob_cols)

# Remove likely identifiers/high-cardinality personal identifiers.
id_like = {
    "trans_num", "transaction_id", "id", "cc_num", "card_number",
    "first", "last", "street", "zip", "unix_time"
}
drop_cols = [c for c in X.columns if c.lower() in id_like]
X = X.drop(columns=drop_cols, errors="ignore")

# Keep numeric and categorical features only; remove empty columns.
X = X.replace([np.inf, -np.inf], np.nan)
X = X.dropna(axis=1, how="all")
for col in X.select_dtypes(include=["object"]).columns:
    # Prevent datetime-like objects from unexpectedly reaching the encoder.
    X[col] = X[col].astype("string").fillna("Missing").astype(str)

# Remove any columns with only one observed value.
constant_cols = [c for c in X.columns if X[c].nunique(dropna=False) <= 1]
X = X.drop(columns=constant_cols)

print("Feature matrix:", X.shape)
print("Dropped identifier columns:", drop_cols)
print("Dropped constant columns:", constant_cols)
print("Fraud cases:", int((y == 1).sum()))
print("Legitimate cases:", int((y == 0).sum()))
display(X.head())


## 5. Split data and build preprocessing pipelines

A stratified split keeps approximately the same fraud proportion in train and test sets. The test set is held back until evaluation. Scaling and encoding are fitted on the training data only, which helps prevent data leakage.


In [ ]:
# 5. Train/test split and reusable preprocessing
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)

numeric_features = X_train.select_dtypes(include=[np.number, "bool"]).columns.tolist()
categorical_features = [c for c in X_train.columns if c not in numeric_features]

numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=10))
])

preprocessor = ColumnTransformer(transformers=[
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
], remainder="drop")

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))
print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))


## 6. Train models

We compare Logistic Regression, Decision Tree, and Random Forest. `class_weight='balanced'` (or its Random Forest equivalent) gives more importance to the minority fraud class. The tree models are configured with limits to keep initial experiments manageable on a laptop.


In [ ]:
# 6. Train and evaluate baseline models
models = {
    "Logistic Regression": Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(
            max_iter=1000, class_weight="balanced", solver="liblinear",
            random_state=RANDOM_STATE
        ))
    ]),
    "Decision Tree": Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("classifier", DecisionTreeClassifier(
            max_depth=12, min_samples_leaf=5, class_weight="balanced",
            random_state=RANDOM_STATE
        ))
    ]),
    "Random Forest": Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("classifier", RandomForestClassifier(
            n_estimators=150, max_depth=18, min_samples_leaf=3,
            class_weight="balanced_subsample", n_jobs=-1,
            random_state=RANDOM_STATE
        ))
    ])
}

fitted_models = {}
results = []
prediction_cache = {}

for name, estimator in models.items():
    print(f"Training {name} ...")
    estimator.fit(X_train, y_train)
    y_pred = estimator.predict(X_test)
    y_score = estimator.predict_proba(X_test)[:, 1]

    row = {
        "model": name,
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred, zero_division=0),
        "f1": f1_score(y_test, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_test, y_score),
        "average_precision_pr_auc": average_precision_score(y_test, y_score)
    }
    results.append(row)
    fitted_models[name] = estimator
    prediction_cache[name] = (y_pred, y_score)
    print(f"  Precision={row['precision']:.4f} | Recall={row['recall']:.4f} | "
          f"F1={row['f1']:.4f} | PR-AUC/AP={row['average_precision_pr_auc']:.4f}")

results_df = pd.DataFrame(results).sort_values(
    "average_precision_pr_auc", ascending=False
).reset_index(drop=True)
display(results_df.round(4))
results_df.to_csv(OUTPUT_DIR / "model_comparison.csv", index=False)


## 7. Evaluate the models

For fraud detection, **recall** measures how many fraudulent transactions were caught, while **precision** measures how many flagged transactions were actually fraud. Average precision (often used as a PR-AUC summary) is useful when fraud is rare. Always inspect the confusion matrix and do not select a model using accuracy alone.


In [ ]:
# 7. Detailed classification reports and confusion matrices
for name, estimator in fitted_models.items():
    y_pred, y_score = prediction_cache[name]
    print("\n" + "=" * 70)
    print(name)
    print(classification_report(
        y_test, y_pred, labels=[0, 1],
        target_names=["Legitimate", "Fraud"], zero_division=0
    ))
    cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
    fig, ax = plt.subplots(figsize=(5.5, 4.5))
    ConfusionMatrixDisplay(
        confusion_matrix=cm, display_labels=["Legitimate", "Fraud"]
    ).plot(ax=ax, values_format="d", colorbar=False)
    ax.set_title(f"{name} — Confusion Matrix")
    plt.tight_layout()
    safe_name = re.sub(r"[^a-z0-9]+", "_", name.lower()).strip("_")
    fig.savefig(OUTPUT_DIR / f"{safe_name}_confusion_matrix.png", dpi=150)
    plt.show()


In [ ]:
# 8. Compare precision-recall curves
fig, ax = plt.subplots(figsize=(8, 6))
for name, (y_pred, y_score) in prediction_cache.items():
    PrecisionRecallDisplay.from_predictions(
        y_test, y_score, name=name, ax=ax
    )
ax.set_title("Precision–Recall Curves")
ax.grid(True, alpha=0.3)
plt.tight_layout()
fig.savefig(OUTPUT_DIR / "precision_recall_curves.png", dpi=150)
plt.show()

fig, ax = plt.subplots(figsize=(8, 6))
for name, (y_pred, y_score) in prediction_cache.items():
    RocCurveDisplay.from_predictions(y_test, y_score, name=name, ax=ax)
ax.set_title("ROC Curves")
ax.grid(True, alpha=0.3)
plt.tight_layout()
fig.savefig(OUTPUT_DIR / "roc_curves.png", dpi=150)
plt.show()


## 8. Try a sample transaction

The sample below uses the feature columns from the loaded dataset and fills unspecified values from a real row. This demonstrates the prediction interface; it is not a guarantee that the selected transaction is fraudulent.


In [ ]:
# 9. Demonstrate a prediction on one held-out transaction
best_model_name = results_df.loc[0, "model"]
best_model = fitted_models[best_model_name]
sample_transaction = X_test.iloc[[0]].copy()
predicted_label = int(best_model.predict(sample_transaction)[0])
fraud_probability = float(best_model.predict_proba(sample_transaction)[0, 1])

print("Selected model:", best_model_name)
print("Predicted class:", "FRAUD" if predicted_label == 1 else "LEGITIMATE")
print(f"Estimated fraud probability: {fraud_probability:.4f}")
display(sample_transaction)


## 9. Save the best model and run metadata

The model with the highest average precision is saved as a baseline selection. Thresholds and model choice should ultimately reflect the cost of missed fraud, false alerts, and the available review capacity.


In [ ]:
# 10. Save the best model and metadata
best_model_name = results_df.loc[0, "model"]
safe_name = re.sub(r"[^a-z0-9]+", "_", best_model_name.lower()).strip("_")
model_path = MODEL_DIR / "credit_card_fraud_model.joblib"

joblib.dump({
    "model": fitted_models[best_model_name],
    "model_name": best_model_name,
    "feature_columns": list(X.columns),
    "target_column": target_col,
    "metrics": results_df.loc[0].to_dict(),
    "random_state": RANDOM_STATE
}, model_path)

metadata = {
    "dataset_path": str(csv_path),
    "rows_loaded": int(len(df)),
    "features_used": int(X.shape[1]),
    "target_column": target_col,
    "class_counts": {str(int(k)): int(v) for k, v in y.value_counts().sort_index().items()},
    "test_size": TEST_SIZE,
    "random_state": RANDOM_STATE,
    "best_model_by_average_precision": best_model_name,
    "metrics": results_df.loc[0].to_dict()
}
(OUTPUT_DIR / "run_summary.json").write_text(json.dumps(metadata, indent=2, default=float), encoding="utf-8")

print("Saved model:", model_path.resolve())
print("Saved metrics:", (OUTPUT_DIR / "model_comparison.csv").resolve())
print("Saved metadata:", (OUTPUT_DIR / "run_summary.json").resolve())


## 10. Conclusion

This notebook trains and compares three supervised learning baselines for credit-card fraud detection. Because fraudulent transactions are rare, prioritize precision, recall, F1-score, average precision, and confusion matrices over accuracy alone.

**Before submitting:** run all cells from top to bottom, confirm the dataset path is correct, review the generated metrics, and include the actual outputs in your report. These results are educational and should not be used as the sole basis for blocking real transactions.
